In [1]:
import requests
import json
import pandas as pd
import folium
import time
from datetime import datetime
import os
from pathlib import Path

url_race='https://sarl.ingenium.net.au/racelog?racenr=38602'
url_boat='http://srv.sailaway.world/cgi-bin/sailaway/APIBoatInfo.pl?usrnr=59528&key=7B79EE2988A44080A37C06570F4B5EE8'
user='Viper Vit'
boat='Petsamo'
html=os.path.join(str(Path.home()), 'Documents', boat + '.html')

In [2]:
now=datetime.now().strftime('%h-%m %H:%M')

In [3]:
user_fleet=requests.get(url_boat)

In [4]:
for each in user_fleet.json():
    name=each['boatname']
    if name==boat:
        lat=round(each['latitude'],3)
        lon=round(each['longitude'],3)
curr_pos=[lat, lon]
curr_pos

[-30.36, 153.647]

In [5]:
res=requests.get(url_race)

In [6]:
boats = res.json()['result']

In [7]:
boat_data = boats[user + '-' + boat]

In [8]:
boat_data

{'heading': 75,
 'lat_dec': -30.5804,
 'lon_dec': 153.7163,
 'lastreport_heading': '75',
 'lastreport_speed': '3.3',
 'sail': '253392',
 'status': None,
 'wind': '30°, 5.7kn.',
 'ubtname': 'Petsamo',
 'usrname': 'Viper Vit',
 'racetime': -1041372000,
 'teamnr': 0,
 'resultdescr': "S30°34.822' E153°42.975', Hdg: 75°, Spd: 3.3kn., 3137.9nm. to mark 2/13",
 'finished': 'false',
 'teaname': None,
 'usrnr': 59528,
 'rank': '10',
 'btptype': "45' Ketch",
 'racing': 1,
 'timestamp': 1672787526000.0,
 'trackdistance': '259.466',
 'points': '0',
 'nextmarknr': '2',
 'distancetonextmark': '3137.92200255046',
 'track': [[-30.5804, 153.7163],
  [-30.7102, 153.6846],
  [-31.2025, 153.6894],
  [-31.553, 153.6067],
  [-31.8249, 153.5687],
  [-31.9825, 153.5311],
  [-32.0573, 153.2082],
  [-32.1097, 152.8282],
  [-32.8147, 152.4267],
  [-32.9422, 152.2165],
  [-33.2838, 152.1145],
  [-33.6843, 152.0237],
  [-33.8319, 151.3993]]}

In [9]:
#print('Last update: {}'.format(time.ctime(boat_data['timestamp'])))

In [10]:
track=boat_data['track']
track.reverse()
track.append(curr_pos)
track

[[-33.8319, 151.3993],
 [-33.6843, 152.0237],
 [-33.2838, 152.1145],
 [-32.9422, 152.2165],
 [-32.8147, 152.4267],
 [-32.1097, 152.8282],
 [-32.0573, 153.2082],
 [-31.9825, 153.5311],
 [-31.8249, 153.5687],
 [-31.553, 153.6067],
 [-31.2025, 153.6894],
 [-30.7102, 153.6846],
 [-30.5804, 153.7163],
 [-30.36, 153.647]]

In [11]:
df_track=pd.DataFrame(track)
df_track.columns=['Lat', 'Lon']
df_track

,Lat,Lon
0,-33.8319,151.3993
1,-33.6843,152.0237
2,-33.2838,152.1145
3,-32.9422,152.2165
4,-32.8147,152.4267
5,-32.1097,152.8282
6,-32.0573,153.2082
7,-31.9825,153.5311
8,-31.8249,153.5687
9,-31.5530,153.6067


In [12]:
list(df_track.index)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

In [13]:
center=(track[0][0], track[0][1])
mymap=folium.Map(location=center, zoom_start=7)
for idx in df_track.index:
    lat=df_track.loc[idx, 'Lat']
    lon=df_track.loc[idx, 'Lon']
    popup=str(lat) + ', ' + str(lon)
    if idx==df_track.index.max():
        popup=now + ' ' + popup
    folium.Marker([lat, lon], popup=popup).add_to(mymap)
mymap

In [14]:
mymap.save(html)